Loading of dataset which has been created

In [2]:
import pandas as pd

users=pd.read_csv("user.csv")
feedback=pd.read_csv("feedback.csv")

users.head()

,Unnamed: 0,user_id,name,age,location,profession,exp_years,professional_summary,about_me,MBTI,intrests
0,0,X01,Aarav,40,Bangalore,Software Engineer,2,Developing scalable backend systems using Pyth...,Passionate about continuous learning and perso...,INTP,"Data, Fitness, Teaching"
1,1,X02,Vivaan,20,Hyderabad,Data Analyst,7,Analyzing business data trends using SQL and P...,Technology enthusiast who enjoys solving real-...,INTJ,"Coding, AI, Gaming"
2,2,X03,Advik,20,Chennai,Business Analyst,14,Gathering business requirements and improving ...,Creative thinker with a love for innovation an...,ENTJ,"Marketing, Travel, Business"
3,3,X04,Reyansh,43,Pune,Project Manager,9,Managing cross-functional teams and ensuring t...,Always curious about new ideas and emerging tr...,ESTJ,"Education, Reading, Writing"
4,4,X05,Shaurya,28,Delhi,Product Manager,3,Defining product strategies and coordinating f...,Dedicated professional who values collaboratio...,ENTJ,"Health, Fitness, Research"


Execution of NLP

In [3]:
import pandas as pd

from sklearn.feature_extraction.text import (
    TfidfVectorizer
)

users = pd.read_csv("user.csv")

users["combined_text"] = (
    users["professional_summary"]
    + " "
    + users["about_me"]
)

vectorizer = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = vectorizer.fit_transform(
    users["combined_text"]
)

Creating Matching Engine

In [4]:
MBTI_SCORES = {
    ("INTJ","ENFP"):100,
    ("ENFP","INTJ"):100,
    ("INFJ","ENTP"):95,
    ("ENTP","INFJ"):95
}

def mbti_score(a,b):

    return MBTI_SCORES.get((a,b),60)

def location_score(a,b):

    if a==b:
        return 100

    return 50


w1=0.5
w2=0.3
w3=0.2

In [5]:
def compatibility(user1,user2):

    idx1 = users[
        users["user_id"]==user1
    ].index[0]

    idx2 = users[
        users["user_id"]==user2
    ].index[0]

    text_score = cosine_similarity(
        tfidf_matrix[idx1],
        tfidf_matrix[idx2]
    )[0][0]*100

    mbti = mbti_score(
        users.loc[idx1,"mbti"],
        users.loc[idx2,"mbti"]
    )

    location = location_score(
        users.loc[idx1,"location"],
        users.loc[idx2,"location"]
    )

    total = (
        w1*text_score +
        w2*mbti +
        w3*location
    )

    return round(total,2)

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

def compatibility(user1,user2):

    idx1 = users[
        users["user_id"]==user1
    ].index[0]

    idx2 = users[
        users["user_id"]==user2
    ].index[0]

    text_score = cosine_similarity(
        tfidf_matrix[idx1],
        tfidf_matrix[idx2]
    )[0][0]*100

    mbti = mbti_score(
        users.loc[idx1,"MBTI"],
        users.loc[idx2,"MBTI"]
    )

    location = location_score(
        users.loc[idx1,"location"],
        users.loc[idx2,"location"]
    )

    total = (
        w1*text_score +
        w2*mbti +
        w3*location
    )

    return round(total,2)

def top_matches(user_id):

    scores=[]

    for uid in users["user_id"]:

        if uid!=user_id:

            score = compatibility(
                user_id,
                uid
            )

            scores.append(
                (uid,score)
            )

    scores.sort(
        key=lambda x:x[1],
        reverse=True
    )

    return scores[:5]


top_matches("X01")

[('X19', np.float64(36.95)),
 ('X31', np.float64(36.2)),
 ('X56', np.float64(35.85)),
 ('X23', np.float64(35.69)),
 ('X35', np.float64(33.87))]

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics.pairwise import cosine_similarity

X=[]
y=[]

for _,row in feedback.iterrows():

    idx1=users[
        users["user_id"]==
        row["user_id"]
    ].index[0]

    idx2=users[
        users["user_id"]==
        row["matched_user_id"]
    ].index[0]

    text_sim=cosine_similarity(
        tfidf_matrix[idx1],
        tfidf_matrix[idx2]
    )[0][0]

    # Fixed: changed 'mbti' to 'MBTI'
    mbti=mbti_score(
        users.loc[idx1,"MBTI"],
        users.loc[idx2,"MBTI"]
    )/100

    X.append([
        text_sim,
        mbti
    ])

    y.append(
        row["action"]
    )


In [8]:
model=LinearRegression()



In [9]:
model.fit(X,y)

print("Learned Weights")

print(
    "Text Similarity:",
    model.coef_[0]
)

print(
    "MBTI:",
    model.coef_[1]
)

Learned Weights
Text Similarity: -0.26214484935105065
MBTI: 0.02759366226880725


In [10]:
# Take User ID from user
user_id = input("Enter User ID: ").strip().upper()

# Check if User ID exists
if user_id in users["user_id"].values:

    print(f"\nTop Matches for {user_id}\n")

    matches = top_matches(user_id)

    for uid, score in matches:
        print(f"{uid} ---> {score}%")

else:
    print("Invalid User ID! Please enter a valid User ID.")

Enter User ID (Example: X01): X04

Top Matches for X04

X66 ---> 37.97%
X44 ---> 35.4%
X60 ---> 34.47%
X24 ---> 34.1%
X29 ---> 33.56%
